In [ ]:
# Phase 4: Data Cleaning & Feature Engineering

**Objective:** 
Transform the raw, messy DataFrame from Phase 3 into a clean, analysis-ready dataset. This includes handling missing values, converting data types, extracting new features, and removing duplicates.

**Business Goal:** 
Prepare a high-quality dataset to accurately calculate our "User Profile Quality Score" and perform regional segmentation.

**Key Activities:**
1. Handle Missing Values (especially `id_value`).
2. Convert string dates to datetime.
3. Extract new features (Email domain, Age groups, etc.).
4. Check for and remove duplicate records.
5. Export the cleaned dataset to `data/processed/cleaned_users.csv`.

In [2]:
# Cell 1: Project Setup (Manual Override - Bulletproof)
import os
import sys
import json
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

# ============================================================
# STEP 1: MANUALLY SET YOUR PROJECT ROOT PATH
# ============================================================
# CHANGE THIS TO YOUR ACTUAL PROJECT PATH:
PROJECT_ROOT = r"C:\Users\Usman Khan\global-user-onboarding-analytics"

# ============================================================
# STEP 2: Verify and Switch to Project Root
# ============================================================
if os.path.exists(PROJECT_ROOT):
    os.chdir(PROJECT_ROOT)
    print(f"✅ Project root set to: {os.getcwd()}")
else:
    print(f"❌ ERROR: Project root not found at: {PROJECT_ROOT}")
    print("Please update the 'PROJECT_ROOT' variable above with your correct path.")
    raise FileNotFoundError("Project root not found!")

# ============================================================
# STEP 3: Verify folder structure exists
# ============================================================
print("\nVerifying folder structure:")
folders_to_check = ["data/raw/", "src/", "notebooks/"]
for folder in folders_to_check:
    exists = os.path.exists(folder)
    status = "✅" if exists else "❌"
    print(f"  {status} {folder}: {exists}")

# ============================================================
# STEP 4: Load the raw JSON data
# ============================================================
print("\nLoading raw data...")
raw_data_folder = "data/raw/"
all_files = os.listdir(raw_data_folder)
json_files = [f for f in all_files if f.endswith('.json')]

if not json_files:
    raise FileNotFoundError("No JSON files found in data/raw/!")

latest_json_file = sorted(json_files)[-1]
file_path = os.path.join(raw_data_folder, latest_json_file)
print(f"  📂 Loading: {latest_json_file}")

with open(file_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# ============================================================
# STEP 5: Flatten into DataFrame
# ============================================================
users_list = raw_data['results']
df = pd.json_normalize(users_list, sep="_")

print(f"\n✅ DataFrame loaded successfully!")
print(f"   Shape: {df.shape}")
print(f"   Columns: {df.shape[1]}")

✅ Project root set to: C:\Users\Usman Khan\global-user-onboarding-analytics

Verifying folder structure:
  ✅ data/raw/: True
  ✅ src/: True
  ✅ notebooks/: True

Loading raw data...
  📂 Loading: raw_users_2026-08-05_12-29-46.json

✅ DataFrame loaded successfully!
   Shape: (5000, 34)
   Columns: 34


In [3]:
# ============================================================
# TASK 1: Handling Missing Values in 'id_value' Column
# ============================================================
# Business Context: 
# 'id_value' contains the user's national ID/passport number.
# We will flag users with missing IDs and fill placeholder values.
# ============================================================

# Step 1: Create a boolean flag column 'has_id'
# True = User has a valid ID, False = User is missing an ID
df["has_id"] = np.where(df["id_value"].isna(), False, True)

# Step 2: Verify the distribution of the new flag
print("--- ID Availability Distribution ---")
id_distribution = df["has_id"].value_counts()
print(id_distribution)

# SAFE WAY: Use .get() to avoid KeyError
# .get() returns None if the key doesn't exist
missing_count = id_distribution.get(False, 0)
print(f"Total users with MISSING ID: {missing_count}")

# Step 3: Replace null values with a clear placeholder string
# Use .fillna() which is cleaner and faster than .loc[]
df['id_value'] = df['id_value'].fillna('MISSING_ID')

# Step 4: Verification - Confirm that NO nulls remain in the column
remaining_nulls = df['id_value'].isna().sum()
print(f"\n--- Verification ---")
print(f"Remaining null values in 'id_value': {remaining_nulls}")
if remaining_nulls == 0:
    print("✅ All nulls handled successfully!")
else:
    print("⚠️ Warning: Nulls still exist!")

# Step 5: Display a sample of the updated column to visually confirm
print(f"\n--- Sample of filled values (first 5 rows with missing IDs) ---")
sample_rows = df.loc[df['has_id'] == False, ['id_value', 'has_id']].head()
if len(sample_rows) > 0:
    print(sample_rows)
else:
    print("No rows with missing IDs found!")

# Step 6: Quick sanity check
print(f"\n--- Sanity Check ---")
print(f"Total rows: {len(df)}")
true_count = (df['has_id'] == True).sum()
false_count = (df['has_id'] == False).sum()
print(f"has_id = True: {true_count}")
print(f"has_id = False: {false_count}")
print(f"Sum: {true_count + false_count}")

--- ID Availability Distribution ---
has_id
True     4069
False     931
Name: count, dtype: int64
Total users with MISSING ID: 931

--- Verification ---
Remaining null values in 'id_value': 0
✅ All nulls handled successfully!

--- Sample of filled values (first 5 rows with missing IDs) ---
     id_value  has_id
0  MISSING_ID   False
1  MISSING_ID   False
4  MISSING_ID   False
6  MISSING_ID   False
8  MISSING_ID   False

--- Sanity Check ---
Total rows: 5000
has_id = True: 4069
has_id = False: 931
Sum: 5000


In [4]:
# ============================================================
# TASK 2: Date Conversion & Feature Engineering
# ============================================================
# Business Context:
# Converting string dates to datetime to extract features
# for segmentation, trend analysis, and business reporting.
# ============================================================

print("=== DATE CONVERSION & FEATURE ENGINEERING ===\n")

# Step 1: Convert strings to datetime
# errors='coerce' handles invalid dates by converting them to NaT
print("1. Converting date columns to datetime...")
df["dob_date"] = pd.to_datetime(df["dob_date"], errors="coerce")
df["registered_date"] = pd.to_datetime(df["registered_date"], errors="coerce")

# Verify the conversion worked (FIX: dttype -> dtype)
print(f"   dob_date type: {df['dob_date'].dtype}")
print(f"   registered_date type: {df['registered_date'].dtype}")
print("   ✅ Date conversion complete!")

# Step 2: Extract features from registered_date
print("\n2. Extracting features from registration date...")
df["registration_year"] = df["registered_date"].dt.year
df["registration_month"] = df["registered_date"].dt.month
df["registration_day_of_week"] = df["registered_date"].dt.dayofweek  # 0=Monday
df["registration_quarter"] = df["registered_date"].dt.quarter
print("   ✅ Registration features created!")

# Step 3: Extract features from date of birth
print("\n3. Extracting features from date of birth...")
df["dob_year"] = df["dob_date"].dt.year
df["dob_month"] = df["dob_date"].dt.month
print("   ✅ DOB features created!")

# Step 4: Create age groups (categorical segmentation)
print("\n4. Creating age group segmentation...")
# Using cascade conditions for cleaner logic
conditions = [
    (df["dob_age"] <= 25),    # 18-25 (catches 25 and under)
    (df["dob_age"] <= 40),    # 26-40
    (df["dob_age"] <= 60),    # 41-60
    (df["dob_age"] > 60)      # 60+
]
choices = ["18-25", "26-40", "41-60", "60+"]
df["age_group"] = np.select(conditions, choices, default="Unknown")

# Check the distribution
age_distribution = df["age_group"].value_counts()
print("   Age group distribution:")
print(age_distribution)
print(f"   ✅ Age groups created!")

# Step 5: Calculate days since registration
print("\n5. Calculating days since registration...")
#  Make today timezone-aware in UTC to match the data
today = pd.Timestamp.now(tz='UTC')
df["days_since_registration"] = (today - df["registered_date"]).dt.days
print(f"   ✅ Days since registration calculated!")

# Step 6: Quick sanity check
print("\n=== SANITY CHECK ===")
new_columns = [
    'registration_year', 'registration_month', 'registration_day_of_week',
    'registration_quarter', 'dob_year', 'dob_month', 'age_group',
    'days_since_registration'
]
for col in new_columns:
    null_count = df[col].isna().sum()
    if null_count == 0:
        print(f"   ✅ {col}: {null_count} nulls")
    else:
        print(f"   ⚠️ {col}: {null_count} nulls (needs attention)")

# Step 7: Display sample of new features
print("\n=== SAMPLE OF NEW FEATURES ===")
print(df[new_columns].head(10))

# Step 8: Additional verification - check days range
print("\n=== DAYS SINCE REGISTRATION STATISTICS ===")
print(f"Min days since registration: {df['days_since_registration'].min()}")
print(f"Max days since registration: {df['days_since_registration'].max()}")
print(f"Mean days since registration: {df['days_since_registration'].mean():.0f}")

=== DATE CONVERSION & FEATURE ENGINEERING ===

1. Converting date columns to datetime...
   dob_date type: datetime64[us, UTC]
   registered_date type: datetime64[us, UTC]
   ✅ Date conversion complete!

2. Extracting features from registration date...
   ✅ Registration features created!

3. Extracting features from date of birth...
   ✅ DOB features created!

4. Creating age group segmentation...
   Age group distribution:
age_group
60+      1880
41-60    1727
26-40    1333
18-25      60
Name: count, dtype: int64
   ✅ Age groups created!

5. Calculating days since registration...
   ✅ Days since registration calculated!

=== SANITY CHECK ===
   ✅ registration_year: 0 nulls
   ✅ registration_month: 0 nulls
   ✅ registration_day_of_week: 0 nulls
   ✅ registration_quarter: 0 nulls
   ✅ dob_year: 0 nulls
   ✅ dob_month: 0 nulls
   ✅ age_group: 0 nulls
   ✅ days_since_registration: 0 nulls

=== SAMPLE OF NEW FEATURES ===
   registration_year  registration_month  registration_day_of_week  \

In [5]:
# ============================================================
# TASK 3: Duplicate Detection & Removal
# ============================================================
# Business Context:
# Duplicate user records can skew metrics, waste marketing spend,
# and create compliance risks. We need to identify and remove them.
# ============================================================

print("\n" + "=" * 60)
print("TASK 3: DUPLICATE DETECTION & REMOVAL")
print("=" * 60 + "\n")

# ============================================================
# STEP 1: Check for email duplicates
# ============================================================
print("1. Checking for duplicate email records...")
email_duplicate_count = df.duplicated(subset=['email']).sum()
print(f"   ✅ Found {email_duplicate_count} duplicate email records to remove")

# ============================================================
# STEP 2: Check for name + city duplicates
# ============================================================
print("\n2. Checking for duplicate name + city combinations...")
name_city_duplicate_count = df.duplicated(subset=['name_first', 'name_last', 'location_city']).sum()
print(f"   ✅ Found {name_city_duplicate_count} duplicate name + city records to remove")

# ============================================================
# STEP 3: Investigate ALL duplicates (professional investigation)
# ============================================================
print("\n3. Investigating duplicate records...")

# 3.1 Email duplicates investigation
if email_duplicate_count > 0:
    total_email_duplicate_rows = df[df['email'].duplicated(keep=False)].shape[0]
    print(f"   📊 Email duplicates:")
    print(f"   - Total rows in email duplicate groups: {total_email_duplicate_rows}")
    print(f"   - Duplicate email records to remove: {email_duplicate_count}")
    
    # Show sample of email duplicates
    print("\n   Sample of email duplicate records (first 5):")
    duplicate_emails = df[df['email'].duplicated(keep=False)]
    display_cols = ['email', 'name_first', 'name_last', 'location_city', 'has_id']
    print(duplicate_emails[display_cols].head(5))
else:
    print("   ✅ No email duplicates found!")

# 3.2 Name+City duplicates investigation
if name_city_duplicate_count > 0:
    name_city_duplicates = df[df.duplicated(subset=['name_first', 'name_last', 'location_city'], keep=False)]
    print(f"\n   📊 Name + City duplicates:")
    print(f"   - Total rows in name+city duplicate groups: {len(name_city_duplicates)}")
    print(f"   - Duplicate name+city records to remove: {name_city_duplicate_count}")
    
    # Show details of name+city duplicates
    print("\n   Details of name+city duplicates:")
    display_cols = ['email', 'name_first', 'name_last', 'location_city', 'email', 'dob_age', 'has_id']
    print(name_city_duplicates[display_cols])
    
    # Check if these are ALSO in email duplicates
    duplicate_emails_set = set(df[df['email'].duplicated(keep=False)]['email'])
    name_city_emails_set = set(name_city_duplicates['email'])
    overlap = duplicate_emails_set.intersection(name_city_emails_set)
    
    if len(overlap) > 0:
        print(f"\n   ⚠️ Note: {len(overlap)} of these name+city duplicates ALSO have duplicate emails")
        print(f"   These will be handled by email duplicate removal.")
    else:
        print(f"\n   ℹ️ Note: These name+city duplicates have UNIQUE emails")
        print(f"   They represent potential duplicates that need special handling.")
else:
    print("   ✅ No name + city duplicates found!")

# ============================================================
# STEP 4: Remove duplicates
# ============================================================
print("\n4. Removing duplicates...")

# 4.1 Remove email duplicates first (priority)
if email_duplicate_count > 0:
    df = df.drop_duplicates(subset=['email'], keep='first')
    print(f"   ✅ Removed {email_duplicate_count} email duplicate users")
else:
    print("   ✅ No email duplicates to remove")

# 4.2 Remove name+city duplicates that are NOT already removed
if name_city_duplicate_count > 0:
    # Check if name+city duplicates are already covered by email duplicates
    name_city_duplicates = df[df.duplicated(subset=['name_first', 'name_last', 'location_city'], keep=False)]
    
    if len(name_city_duplicates) > 0:
        # We still have name+city duplicates to handle
        df = df.drop_duplicates(subset=['name_first', 'name_last', 'location_city'], keep='first')
        print(f"   ✅ Removed {name_city_duplicate_count} name+city duplicate users")
    else:
        print(f"   ✅ Name+city duplicates were already handled by email duplicate removal")
else:
    print("   ✅ No name+city duplicates to remove")

# ============================================================
# STEP 5: Final verification
# ============================================================
print(f"\n5. Final verification:")
print(f"   New DataFrame shape: {df.shape[0]} rows × {df.shape[1]} columns")

# Calculate expected rows
expected_rows = 5000 - email_duplicate_count - (name_city_duplicate_count if len(name_city_duplicates) > 0 else 0)
if df.shape[0] == expected_rows:
    print(f"   ✅ Shape is correct!")
else:
    print(f"   ℹ️ Note: Expected ~{expected_rows} rows, got {df.shape[0]} rows")
    print(f"   This is because some name+city duplicates were already removed by email deduplication")

# ============================================================
# STEP 6: Sanity check
# ============================================================
remaining_email_duplicates = df.duplicated(subset=['email']).sum()
remaining_name_city_duplicates = df.duplicated(subset=['name_first', 'name_last', 'location_city']).sum()

print(f"\n6. Sanity check:")
print(f"   Remaining email duplicates: {remaining_email_duplicates}")
print(f"   Remaining name+city duplicates: {remaining_name_city_duplicates}")

if remaining_email_duplicates == 0 and remaining_name_city_duplicates == 0:
    print("   ✅ No duplicates remain! Dataset is clean.")
else:
    print(f"   ⚠️ Warning: Duplicates still exist!")

# ============================================================
# STEP 7: Summary Report
# ============================================================
print("\n" + "=" * 60)
print("📊 DUPLICATE REMOVAL SUMMARY")
print("=" * 60)
print(f"Original records: 5,000")
print(f"Email duplicates removed: {email_duplicate_count}")
print(f"Name+City duplicates removed: {name_city_duplicate_count}")
print(f"Final records: {df.shape[0]}")
print(f"Clean dataset: {'✅ YES' if remaining_email_duplicates == 0 and remaining_name_city_duplicates == 0 else '⚠️ WARNING'}")
print("=" * 60)
print("✅ TASK 3 COMPLETE")
print("=" * 60)


TASK 3: DUPLICATE DETECTION & REMOVAL

1. Checking for duplicate email records...
   ✅ Found 38 duplicate email records to remove

2. Checking for duplicate name + city combinations...
   ✅ Found 1 duplicate name + city records to remove

3. Investigating duplicate records...
   📊 Email duplicates:
   - Total rows in email duplicate groups: 76
   - Duplicate email records to remove: 38

   Sample of email duplicate records (first 5):
                               email  name_first name_last location_city  \
38             john.king@example.com        John      King      Hamilton   
70            mhmdth.prs@example.com    محمدطاها     پارسا         اهواز   
134            lyn.mwswy@example.com       الینا     موسوی        سیرجان   
193         leon.sanchez@example.com        Léon   Sanchez      Nanterre   
438  stanislava.miskovic@example.com  Stanislava  Mišković         Dečan   

     has_id  
38    False  
70    False  
134   False  
193    True  
438    True  

   📊 Name + City du

## Task 4: Email Domain Analysis (Data Limitation)

**Limitation:** The Random User Generator API provides anonymized data with placeholder email domains (`example.com`). This is standard practice for privacy-protected datasets.

**Impact:** We cannot perform meaningful email domain categorization (disposable, corporate, mainstream) in this project.

**Next Steps:** We'll document this limitation and move to Task 5 (Country Segmentation), which provides valuable business insights.

In [37]:
# ============================================================
# TASK 4: Email Domain Analysis (Documentation Only)
# ============================================================
# NOTE: The Random User Generator API returns anonymized emails
# with 'example.com' domain. No meaningful domain analysis possible.
# ============================================================

print("\n" + "=" * 60)
print("TASK 4: EMAIL DOMAIN ANALYSIS (SKIPPED)")
print("=" * 60 + "\n")

print("📌 DATA NOTE:")
print("   - The API returns anonymized email addresses")
print("   - All emails use the placeholder domain: 'example.com'")
print("   - Real email domain analysis is not possible")
print("   - In production, we would analyze real domains for:")

print("\n   📊 Business Value of Email Domain Analysis:")
print("   ✅ Identify fraudulent/disposable email providers")
print("   ✅ Segment users by email provider (Gmail, Yahoo, etc.)")
print("   ✅ Target corporate users (B2B marketing)")
print("   ✅ Flag high-risk privacy-focused domains (ProtonMail, etc.)")

# Show sample of emails (anonymized)
print("\n   Sample of anonymized emails:")
print(df['email'].head(5).to_string(index=False))

print("\n   Example of what real domains would look like:")
print("   ✅ john.doe@gmail.com → Mainstream")
print("   ✅ jane.smith@company.com → Corporate")
print("   ✅ fraud@mailinator.com → Disposable (⚠️ KYC Risk)")

print("\n✅ Task 4 complete - Moving to Task 5 (Country Segmentation)")


TASK 4: EMAIL DOMAIN ANALYSIS (SKIPPED)

📌 DATA NOTE:
   - The API returns anonymized email addresses
   - All emails use the placeholder domain: 'example.com'
   - Real email domain analysis is not possible
   - In production, we would analyze real domains for:

   📊 Business Value of Email Domain Analysis:
   ✅ Identify fraudulent/disposable email providers
   ✅ Segment users by email provider (Gmail, Yahoo, etc.)
   ✅ Target corporate users (B2B marketing)
   ✅ Flag high-risk privacy-focused domains (ProtonMail, etc.)

   Sample of anonymized emails:
  kubra.erbay@example.com
    prhm.hmdy@example.com
laurie.knight@example.com
 bettina.behr@example.com
    aynz.rdyy@example.com

   Example of what real domains would look like:
   ✅ john.doe@gmail.com → Mainstream
   ✅ jane.smith@company.com → Corporate
   ✅ fraud@mailinator.com → Disposable (⚠️ KYC Risk)

✅ Task 4 complete - Moving to Task 5 (Country Segmentation)


## Task 5: Country Segmentation & Regional Analysis

**Objective:** Analyze the geographical distribution of our user base to inform marketing strategy, product localization, and compliance efforts.

**Business Context:** 
Understanding *where* our users come from is critical for:
- **Marketing Budget Allocation:** Prioritize ad spend in countries with the highest user concentration.
- **Product Localization:** Decide which languages and currencies to support first.
- **Compliance & Risk:** Identify regions with higher rates of incomplete profiles (missing IDs), which may require enhanced KYC verification.
- **Growth Opportunities:** Spot emerging markets that are underrepresented compared to global averages.

**Key Activities:**
1. Identify the top 10 countries by user count and calculate their percentages.
2. Flag users from the top 5 countries for targeted marketing campaigns.
3. Analyze data quality by country (specifically, where are missing IDs most common?).
4. Group countries into broader regions (Americas, Europe, Asia, etc.) to analyze macro-trends.
5. Assess regional data quality to identify which areas need the most operational support.

**Expected Outputs:**
- A clear ranking of countries by user volume.
- A `region` column added to the DataFrame.
- Identification of high-risk regions with high ID-missing rates.

In [98]:
# ============================================================
# TASK 5: Country Segmentation & Regional Analysis
# ============================================================
# Business Context:
# Understanding user geography helps with marketing budget
# allocation, product localization, and identifying growth markets.
# ============================================================


print("TASK 5: COUNTRY SEGMENTATION & REGIONAL ANALYSIS")


# ============================================================
# STEP 1: Check if location_country exists
# ============================================================
print("1. Checking for location data...")
if 'location_country' in df.columns:
    print("   ✅ location_country column found!")
else:
    print("   ❌ location_country not found! Available columns:")
    print(df.columns.tolist())

# ============================================================
# STEP 2: Country Distribution
# ============================================================
print("\n2. Country distribution:")
country_counts = df['location_country'].value_counts()
total_users = len(df)
unique_countries = df['location_country'].nunique()

print(f"   Total users: {total_users}")
print(f"   Unique countries: {unique_countries}")

print("\n   Top 10 countries by user count:")
for i, (country, count) in enumerate(country_counts.head(10).items(), 1):
    percentage = (count / total_users) * 100
    print(f"   {i:2}. {country}: {count} ({percentage:.1f}%)")

# ============================================================
# STEP 3: Create Top Country Flag
# ============================================================
print("\n3. Flagging users from top 5 countries...")
top_countries = country_counts.head(5).index.tolist()
df['is_top_country'] = df['location_country'].isin(top_countries)

top_country_users = df['is_top_country'].sum()
print(f"   Top 5 countries: {', '.join(top_countries)}")
print(f"   Users from top countries: {top_country_users}")
print(f"   Percentage: {(top_country_users / total_users * 100):.1f}%")

# ============================================================
# STEP 4: Country-Wide Data Quality Analysis
# ============================================================
print("\n4. Analyzing data quality by country...")
country_id_quality = df.groupby('location_country').agg({
    'id_value': lambda x: (x == 'MISSING_ID').sum(),
    'has_id': lambda x: (~x).sum()  # Count users without IDs
}).rename(columns={'id_value': 'missing_ids', 'has_id': 'users_without_ids'})

# Add user count and missing percentage
country_id_quality['total_users'] = country_counts
country_id_quality['missing_percentage'] = (country_id_quality['missing_ids'] / country_counts * 100).round(1)

print("   Top 10 countries with highest ID missing rate:")
print("   Country | Missing IDs | Total Users | Missing %")
print("   " + "-" * 50)
for country in country_id_quality.sort_values('missing_percentage', ascending=False).head(10).index:
    row = country_id_quality.loc[country]
    print(f"   {country:12} | {row['missing_ids']:6} | {row['total_users']:8} | {row['missing_percentage']:6.1f}%")

# ============================================================
# STEP 5: Regional Segmentation
# ============================================================
print("\n5. Creating regional segmentation...")

def get_region(country):
    """Map country to region"""
    # Americas
    if country in ['US', 'Canada', 'Mexico', 'Brazil', 'Argentina', 'Chile', 'Colombia', 'Peru', 'Venezuela']:
        return 'Americas'
    # Europe
    elif country in ['UK', 'Germany', 'France', 'Spain', 'Italy', 'Netherlands', 'Sweden', 
                     'Norway', 'Denmark', 'Finland', 'Poland', 'Greece', 'Portugal', 'Switzerland',
                     'Belgium', 'Austria', 'Ireland', 'Czech Republic', 'Hungary']:
        return 'Europe'
    # Asia
    elif country in ['China', 'India', 'Japan', 'South Korea', 'Singapore', 'Malaysia', 
                     'Thailand', 'Vietnam', 'Philippines', 'Indonesia', 'Pakistan', 
                     'Bangladesh', 'Sri Lanka']:
        return 'Asia'
    # Oceania
    elif country in ['Australia', 'New Zealand']:
        return 'Oceania'
    # Africa
    elif country in ['South Africa', 'Nigeria', 'Egypt', 'Kenya', 'Ghana', 'Morocco']:
        return 'Africa'
    else:
        return 'Other'

df['region'] = df['location_country'].apply(get_region)

# Show regional distribution
print("   Regional distribution:")
region_counts = df['region'].value_counts()
for region, count in region_counts.items():
    percentage = (count / total_users) * 100
    print(f"   - {region}: {count} ({percentage:.1f}%)")

# ============================================================
# STEP 6: Regional Data Quality
# ============================================================
print("\n6. Data quality by region:")
region_id_quality = df.groupby('region').agg({
    'id_value': lambda x: (x == 'MISSING_ID').sum(),
    'has_id': lambda x: (~x).sum()
}).rename(columns={'id_value': 'missing_ids', 'has_id': 'users_without_ids'})

region_id_quality['total_users'] = df.groupby('region').size()
region_id_quality['missing_percentage'] = (region_id_quality['missing_ids'] / region_id_quality['total_users'] * 100).round(1)

print("   Region | Missing IDs | Total Users | Missing %")
print("   " + "-" * 50)
for region in region_id_quality.index:
    row = region_id_quality.loc[region]
    print(f"   {region:10} | {row['missing_ids']:6} | {row['total_users']:8} | {row['missing_percentage']:6.1f}%")

# ============================================================
# STEP 7: Final Summary
# ============================================================
print("\n" + "=" * 60)
print("📊 COUNTRY SEGMENTATION SUMMARY")
print("=" * 60)
print(f"Total users: {total_users}")
print(f"Countries represented: {unique_countries}")
print(f"Top country: {country_counts.index[0]} ({country_counts.iloc[0]} users, {country_counts.iloc[0]/total_users*100:.1f}%)")
print(f"Top 5 countries cover: {top_country_users} users ({top_country_users/total_users*100:.1f}%)")
print(f"Largest region: {region_counts.index[0]} ({region_counts.iloc[0]} users, {region_counts.iloc[0]/total_users*100:.1f}%)")

# Find the region with highest missing ID rate
highest_missing_region = region_id_quality.sort_values('missing_percentage', ascending=False).index[0]
print(f"Region with highest ID missing rate: {highest_missing_region} ({region_id_quality.loc[highest_missing_region, 'missing_percentage']:.1f}%)")

print("=" * 60)
print("✅ TASK 5 COMPLETE")
print("=" * 60)

TASK 5: COUNTRY SEGMENTATION & REGIONAL ANALYSIS
1. Checking for location data...
   ✅ location_country column found!

2. Country distribution:
   Total users: 4962
   Unique countries: 21

   Top 10 countries by user count:
    1. Mexico: 256 (5.2%)
    2. Switzerland: 256 (5.2%)
    3. Ukraine: 252 (5.1%)
    4. Norway: 252 (5.1%)
    5. Finland: 247 (5.0%)
    6. United Kingdom: 247 (5.0%)
    7. Ireland: 247 (5.0%)
    8. India: 245 (4.9%)
    9. Canada: 242 (4.9%)
   10. Germany: 242 (4.9%)

3. Flagging users from top 5 countries...
   Top 5 countries: Mexico, Switzerland, Ukraine, Norway, Finland
   Users from top countries: 1263
   Percentage: 25.5%

4. Analyzing data quality by country...
   Top 10 countries with highest ID missing rate:
   Country | Missing IDs | Total Users | Missing %
   --------------------------------------------------
   Iran         |  210.0 |    210.0 |  100.0%
   Ukraine      |  252.0 |    252.0 |  100.0%
   Turkey       |  238.0 |    238.0 |  100.0%
 

## Task 6: Export Cleaned Dataset

**Objective:** Save the fully cleaned and feature-engineered DataFrame to a CSV file for sharing, reporting, and Phase 5 analysis.

**Business Context:** 
Exporting the final dataset is the "delivery" step of the data cleaning phase. This CSV file will serve as the single source of truth for:
- **Product Team:** Understanding user demographics and regions.
- **Marketing Team:** Identifying top countries and disposable email users for campaign targeting.
- **Compliance Team:** Flagging users with missing IDs for enhanced KYC verification.

**Key Activities:**
1. Create the `data/processed/` directory (if it doesn't exist).
2. Export the DataFrame to CSV with a timestamp.
3. Verify the export by checking the file exists and loading it back to confirm no data loss.
4. Print a summary of the final dataset (shape, columns, key statistics).

**Expected Outputs:**
- `cleaned_users_YYYY-MM-DD.csv` in the `data/processed/` folder.
- A summary report printed in the notebook confirming the dataset is ready for Phase 5.

In [103]:
# ============================================================
# TASK 6: Export Cleaned Dataset
# ============================================================
# Business Context:
# Saving the cleaned, feature-engineered dataset to CSV ensures
# we have a reusable "source of truth" for all downstream
# analysis, reporting, and stakeholder sharing.
# ============================================================


print("TASK 6: EXPORT CLEANED DATASET")
print("=" * 60 + "\n")

# ============================================================
# STEP 1: Verify the DataFrame is clean
# ============================================================
print("1. Verifying final dataset quality...")
print(f"   Total rows: {len(df)}")
print(f"   Total columns: {len(df.columns)}")
print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Quick quality checks
missing_values = df.isna().sum().sum()
print(f"   Total missing values remaining: {missing_values}")
if missing_values == 0:
    print("   ✅ No missing values found! Dataset is clean.")
else:
    print(f"   ⚠️ Warning: {missing_values} missing values still exist.")

duplicate_count = df.duplicated(subset=['email']).sum()
print(f"   Duplicate email records remaining: {duplicate_count}")
if duplicate_count == 0:
    print("   ✅ No duplicates found! Dataset is unique.")

# ============================================================
# STEP 2: Create the processed folder (if it doesn't exist)
# ============================================================
print("\n2. Creating output directory...")
output_dir = "data/processed/"
os.makedirs(output_dir, exist_ok=True)
print(f"   ✅ Directory ready: {output_dir}")

# ============================================================
# STEP 3: Generate timestamp for the filename
# ============================================================
print("\n3. Generating timestamp...")
from datetime import datetime
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
filename = f"cleaned_users_{timestamp}.csv"
filepath = os.path.join(output_dir, filename)
print(f"   Filename: {filename}")

# ============================================================
# STEP 4: Export to CSV
# ============================================================
print("\n4. Exporting to CSV...")
df.to_csv(filepath, index=False, encoding='utf-8')
print(f"   ✅ CSV file saved successfully!")
print(f"   📂 File path: {filepath}")

# ============================================================
# STEP 5: Verify the export
# ============================================================
print("\n5. Verifying export...")

# Check if file exists
if os.path.exists(filepath):
    file_size = os.path.getsize(filepath) / 1024  # Size in KB
    print(f"   ✅ File exists! Size: {file_size:.2f} KB")
    
    # Try loading it back to confirm it's readable
    df_verify = pd.read_csv(filepath)
    print(f"   ✅ File loaded successfully!")
    print(f"   Loaded shape: {df_verify.shape}")
    
    # Compare the loaded file with the original
    if df_verify.shape == df.shape:
        print(f"   ✅ Shape matches! Export is complete and correct.")
    else:
        print(f"   ⚠️ Warning: Shape mismatch!")
else:
    print(f"   ❌ ERROR: File not found!")

# ============================================================
# STEP 6: Summary Report
# ============================================================
print("\n" + "=" * 60)
print("📊 EXPORT SUMMARY")
print("=" * 60)
print(f"Final dataset shape: {df.shape}")
print(f"Export format: CSV")
print(f"Export location: {filepath}")
print(f"Number of features: {df.shape[1]}")
print(f"Number of users: {df.shape[0]}")
print("\nKey columns added during cleaning:")
new_columns = ['has_id', 'registration_year', 'registration_month', 'registration_day_of_week',
               'registration_quarter', 'dob_year', 'dob_month', 'age_group', 
               'days_since_registration', 'is_top_country', 'region', 'email_domain']
existing_new_columns = [col for col in new_columns if col in df.columns]
for col in existing_new_columns:
    print(f"   - {col}")

print("\n✅ TASK 6 COMPLETE")
print(f"✅ Dataset is ready for Phase 5: Exploratory Data Analysis (EDA) & Insights!")
print("=" * 60)


TASK 6: EXPORT CLEANED DATASET

1. Verifying final dataset quality...
   Total rows: 4962
   Total columns: 46
   Memory usage: 10.57 MB
   Total missing values remaining: 0
   ✅ No missing values found! Dataset is clean.
   Duplicate email records remaining: 0
   ✅ No duplicates found! Dataset is unique.

2. Creating output directory...
   ✅ Directory ready: data/processed/

3. Generating timestamp...
   Filename: cleaned_users_2026-08-06_16-09-29.csv

4. Exporting to CSV...
   ✅ CSV file saved successfully!
   📂 File path: data/processed/cleaned_users_2026-08-06_16-09-29.csv

5. Verifying export...
   ✅ File exists! Size: 3378.46 KB
   ✅ File loaded successfully!
   Loaded shape: (4962, 46)
   ✅ Shape matches! Export is complete and correct.

📊 EXPORT SUMMARY
Final dataset shape: (4962, 46)
Export format: CSV
Export location: data/processed/cleaned_users_2026-08-06_16-09-29.csv
Number of features: 46
Number of users: 4962

Key columns added during cleaning:
   - has_id
   - registra